# Les differents types de système de recommandation:

# Introduction:

# Nettoyage des données:

In [ ]:
import pandas as pd 
import numpy as np
import random
random.seed(9001)
#pour avoir toujours les memes erreurs à chaque fois qu'on re exécute le projet.

In [ ]:
tab = pd.read_csv('ratings.csv',nrows=100001)


In [ ]:
tab.head()

In [ ]:
useri,frequsers=np.unique(tab.userId,return_counts=True)#useri les id des users, frequsers les freq de chaque user
itemi,freqitems=np.unique(tab.movieId,return_counts=True)#itemi les id des item, freqitem les freq de chaque item
n_users=len(useri)
n_items=len(itemi)
print("le nombre des utilisateurs est :"+ str(n_users) + " Et le nombre des items est: "+ str(n_items))

In [ ]:


indice_user = pd.DataFrame()
indice_user["indice"]=range(1,len(useri)+1)
indice_user["useri"]=useri


indice_item = pd.DataFrame()
indice_item["indice"]=range(1,len(itemi)+1)
indice_item["itemi"]=itemi

In [ ]:
#créer user_ID_new et Item_ID_new
x=[]
y=[]
for i in range(0,len(tab)):
    x.append((indice_user.indice[indice_user.useri==tab.userId[i]].axes[0]+1)[0])
    y.append((indice_item.indice[indice_item.itemi==tab.movieId[i]].axes[0]+1)[0])


In [ ]:
tab["User_ID_new"]=x
tab["Item_ID_new"]=y


In [ ]:



from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold

from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(
    tab[["User_ID_new", "Item_ID_new", "rating"]],
    test_size=0.25,
    random_state=123
)


In [ ]:
sparsity=round(1.0-len(tab)/float(n_users*n_items),3)
print('The sparsity level of our data base is ' +  str(sparsity*100) + '%')

# 1.le Memory based Collaboratif Filtering:

### 1.1 La mise en place du modèle:

In [ ]:

train_data_matrix = np.zeros((n_users, n_items))#matrice nulle de longuer tous les users et tous les items
for line in train_data.itertuples():#parcourire la ligne col par col
    train_data_matrix[line[1]-1, line[2]-1] = line[3] 

test_data_matrix = np.zeros((n_users, n_items))
for line in test_data.itertuples():
    test_data_matrix[line[1]-1, line[2]-1] = line[3]

In [ ]:
#calcule de la cos similarity : (construction du modèle)
from sklearn.metrics.pairwise import pairwise_distances
user_similarity = pairwise_distances(train_data_matrix, metric='cosine')
item_similarity = pairwise_distances(train_data_matrix.T, metric='cosine')
user_similarity1 = pairwise_distances(train_data_matrix, metric='cityblock')
item_similarity1 = pairwise_distances(train_data_matrix.T, metric='cityblock')


In [ ]:
def predict(ratings, similarity, type='user'):#prend
    if type == 'user':
        mean_user_rating = ratings.mean(axis=1)#mean pour chauqe utilisateur (type = float)
        #np.newaxis pour convertir mean_user_rating de array de float en array d'array pour l'utiliser avec ratings
        #puis on a normalisé la var ratings (rating - E)
        ratings_diff = (ratings - mean_user_rating[:, np.newaxis]) #(type === array comme la var rating)
        pred = mean_user_rating[:, np.newaxis] + similarity.dot(ratings_diff) / np.array([np.abs(similarity).sum(axis=1)]).T
    elif type == 'item':
        pred = ratings.dot(similarity) / np.array([np.abs(similarity).sum(axis=1)]) 
        
    x = np.zeros((n_users, n_items))
    for i in range(0,n_items):
        a=max(pred[:,i])
        b=min(pred[:,i])
        c=0
        d=5
        for j in range(0,n_users):
            x[j,i]=(pred[:,i][j]-(a-c))*d/(b-a+c)
    
    return x

In [ ]:
#la prédiction avec les differents modèles:
item_prediction = predict(test_data_matrix, item_similarity, type='item')
user_prediction = predict(test_data_matrix, user_similarity, type='user')
item_prediction1 = predict(test_data_matrix, item_similarity1, type='item')
user_prediction1 = predict(test_data_matrix, user_similarity1, type='user')

### 1.2. La comparaison des RMSE:

In [ ]:
#la creation de la fonction qui calcule le RMSE:
from sklearn.metrics import mean_squared_error
from math import sqrt
def rmse(prediction, ground_truth): #Root Mean Squared Error
    prediction = prediction[ground_truth.nonzero()].flatten() 
    #.flatten() fusionne les elts des array en un array
    #on attribue a prediction, les résultats des prédictions où on connait le vrais rating cad:
    #prediction: tous nos prédictions sur test; ground_truth.nonzero():les vrais résultats qu'on a dans test
    #on va mettre dans prediction les valeurs qu'on a prédit pour les elts qu'on adéja.
    ground_truth = ground_truth[ground_truth.nonzero()].flatten()#pareil dans ground truth
    return sqrt(mean_squared_error(prediction, ground_truth))

In [ ]:
print ('User-based CF RMSE: ' + str(rmse(user_prediction, test_data_matrix)))
print ('Item-based CF RMSE: ' + str(rmse(item_prediction, test_data_matrix)))
print ('User-based1 CF RMSE: ' + str(rmse(user_prediction1, test_data_matrix)))
print ('Item-based1 CF RMSE: ' + str(rmse(item_prediction1, test_data_matrix)))     

### 1.3 Simple comparaison des résultats prédits:

In [ ]:
#comparaison des prediction de modele  user avec l'oeil:
R = pd.DataFrame(test_data_matrix)
R_pred=pd.DataFrame(predict(test_data_matrix, item_similarity1, type='item'))
# Compare true ratings of item 17 with predictions
ratings = pd.DataFrame(data=R.T.loc[16,R.T.loc[16,:] > 0]).head(n=10)
ratings['Prediction'] = R_pred.T.loc[16,R.T.loc[16,:] > 0]
ratings.columns = ['Actual Rating', 'Predicted Rating']
ratings

### 1.4 La généralisation de notre meilleur modèle:

In [ ]:

#creation du modèle finale:
data_matrix = np.zeros((n_users, n_items))
for line in tab[["User_ID_new","Item_ID_new","rating"]].itertuples():#parcourire la ligne col par col
    data_matrix[line[1]-1, line[2]-1] = line[3] 
R = pd.DataFrame(data_matrix)
model_user = pairwise_distances(R, metric='cityblock')#meilleur modèle item-citoyblock .T psk c'est item
model_item = pairwise_distances(R.T, metric='cityblock')#meilleur modèle item-citoyblock .T psk c'est item

R_pred_us=pd.DataFrame(predict(data_matrix, model_user, 'user'))#la prediction
R_pred_it=pd.DataFrame(predict(data_matrix, model_item, 'item'))#la prediction


In [ ]:
def getrecom_membased_for_user( iduser,n=10,ch="all"):
    estim=R_pred_us.loc[iduser-1,R.loc[iduser-1,:] == 0]
    donne=R.loc[iduser-1,R.loc[iduser-1,:] > 0]
    if ch=="discover":
        res=estim
    else:
        res=estim.append(donne)
    res=res.sort_values( ascending=False)[0:n]
    return res

In [ ]:

def getrecom_membased_for_item(iditem, n=10, ch="all"):
    estim = R_pred_it.T.loc[iditem-1, R.T.loc[iditem-1, :] == 0]
    donne = R.T.loc[iditem-1, R.T.loc[iditem-1, :] > 0]

    if ch == "discover":
        res = estim
    else:
        res = pd.concat([estim, donne])

    res = res.sort_values(ascending=False).head(n)
    return res

In [ ]:
(getrecom_membased_for_item(6373,10,"all"))

In [ ]:
(getrecom_membased_for_item(730,10,"discover"))#correcte!

# 2. Model-based Collaborative Filtering

## 2.1 Singular value decomposition (SVD)

### 2.1.1 La mise en place des SVD:

In [ ]:
import scipy.sparse as sp
from scipy.sparse.linalg import svds

#Obtenir les composantes de SVD à partir de la matrice User-Item du train. On choisit une valeur de k=20.
u, s, vt = svds(train_data_matrix, k = 20)
s_diag_matrix=np.diag(s)

In [ ]:
# Multiplication des 3 matrices avec np.dot pour obtenir la matrice User_Item estimée.
X_pred = np.dot(np.dot(u, s_diag_matrix), vt) 

In [ ]:
#la normalisation de X_pred vu qu'elle retourne des données qui sont pas bien distribué dans [0,5]
import math
x = np.zeros((n_users, n_items))
for i in range(0,n_items):
    a=max(X_pred[:,i])
    b=min(X_pred[:,i])
    c=0
    d=5
    for j in range(0,n_users):
        x[j,i]=(X_pred[:,i][j]-(a-c))*d/(b-a+c)
        if math.isnan(x[j,i]): x[j,i]=0

In [ ]:
# Calcul de performance avec RMSE entre la matrice estimée et la matrice du test
print ('RMSE: ' + str(rmse(x, test_data_matrix)))
#On a trouvé 1.49 comme RMSE, c'est plus grand que le RMSE des modèles Memory based, mais ça prend énormement moins du temps.
#Ce qu'on va dans la partie qui suit c'est d'améliorer notre modèle par le gradient stochastique et l'ALS.

### 2.1.2. La généralisation du modele:

In [ ]:
#Après avoir implementer et tester notre modele, nous avons implémenté la fonction getrecom_membased() qui permet de recommander des items 
#à un utilisateur. 
#créant le modèle sur tout le jeu des données:
u, s, vt = svds(data_matrix, k = 20)
s_diag_matrix=np.diag(s)
X_pred = np.dot(np.dot(u, s_diag_matrix), vt) 
import math
x = np.zeros((n_users, n_items))
for i in range(0,n_items):
    a=max(X_pred[:,i])
    b=min(X_pred[:,i])
    c=0
    d=5
    for j in range(0,n_users):
        x[j,i]=(X_pred[:,i][j]-(a-c))*d/(b-a+c)
        if math.isnan(x[j,i]): x[j,i]=0
        

In [ ]:
x=pd.DataFrame(x)

In [ ]:
def getrecom_modbased_svd(iduser, n=10, ch="all"):
    user_ratings = R.loc[iduser-1]
    pred_ratings = x.loc[iduser-1]

    estim = pred_ratings[user_ratings == 0]
    donne = user_ratings[user_ratings > 0]

    res = estim if ch == "discover" else pd.concat([estim, donne])

    return res.sort_values(ascending=False).head(n)

In [ ]:
#recommander l'utilisateur 18 en prenons en compte tous les items y compris ceux qu'il a déjà noté. Afficher les 10 premiers résultats
getrecom_modbased_svd(80,10,"all")

In [ ]:
#recommander l'utilisateur 18 en prenons en compte que les items qu'il n'a pas noté. Afficher les 10 premiers résultats
getrecom_modbased_svd(18,10,"discover")

## 2.2 Stochastic Gradient Descent with Weighted Lambda Regularisation (SGD-WR)

### 2.2.1. La mise en place du modèle:

In [ ]:
#Les matrices I et I2 serviront de matrices de sélecteur pour prendre les éléments appropriés après la création du Train et du Test
#selecteur de var est égal à 1 si la valeur dans la matrice est != 0

# matrice d'indices pour le train
I = train_data_matrix.copy()
I[I > 0] = 1
I[I == 0] = 0

# matrice d'indices pour le test
I2 = test_data_matrix.copy()
I2[I2 > 0] = 1
I2[I2 == 0] = 0

In [ ]:
# La fonction prediction permet de prédire les ratings inconnus en multipliant les matrices P et la transposée de Q
def prediction(P,Q):
    return np.dot(P.T,Q)

In [ ]:

# ****** Initialisation ******* 

lmbda = 0.1 # Terme de régularisation
k = 20 # dimension de l'espace des caractères cachés
m, n = train_data_matrix.shape  # nombre d'utilisateurs et d'items
steps = 150  # Nombre d'itération 
gamma=0.001  # vitesse d'apprentissage

P = 3 * np.random.rand(k,m) # Matrice des caractères cachés pour les utilisateurs
Q = 3 * np.random.rand(k,n) # Matrice des caractères cachés pour les items

#les matrices P et Q sont initialisées avec des valeurs aléatoires au début, mais leur contenu change à chaque itération en se 
#basant sur le train

In [ ]:
""" Il existe plusieurs métriques d'évaluation, mais la plus populaire des métriques utilisée pour évaluer l'exactitude des ratings prédits
est l'erreur quadratique moyenne (RMSE) qu'on a utilisé dans notre projet :

RMSE =RacineCarrée{(1/N) * sum (r_i -estimé{r_i})^2}
"""

def rmse2(I,R,Q,P):
    return np.sqrt(np.sum((I * (R - prediction(P,Q)))**2)/len(R[R > 0]))

#R = train_data_matrix
#prediction(P,Q): estimateur du train_data_matrix avec la méthode de factorisation
#I pour prendre seulement la partie significative de la matrice (!=0)

In [ ]:
#On ne considère que les valeurs !=0 
users,items = train_data_matrix.nonzero()  

In [ ]:
train_errors = []   # RMSE train à chaque step
test_errors = []    # RMSE test à chaque step

for step in range(steps):
    for u, i in zip(users, items):   # tuples (user,item)
        e = train_data_matrix[u, i] - prediction(P[:, u], Q[:, i])

        # mise à jour SGD avec régularisation
        P[:, u] += gamma * (e * Q[:, i] - lmbda * P[:, u])
        Q[:, i] += gamma * (e * P[:, u] - lmbda * Q[:, i])

    train_rmse = rmse2(I, train_data_matrix, Q, P)
    test_rmse  = rmse2(I2, test_data_matrix, Q, P)

    train_errors.append(train_rmse)
    test_errors.append(test_rmse)

In [ ]:
print ('RMSE : ' + str(np.mean(test_errors))) #RMSE = 1.18 ce qui est super! en le comparant avec les autre RMSE (1.5)

In [ ]:
# Maitenant, après avoir obtenus toutes les valeurs de l'erreur à chaque étape,on peut tracer la courbe d'apprentissage.
# ==> On Vérifie la performance en traçant les erreurs du train et du test

import matplotlib.pyplot as plt
%matplotlib inline

plt.plot(range(steps), train_errors, marker='o', label='Training Data'); 
plt.plot(range(steps), test_errors, marker='v', label='Test Data');
plt.title('Courbe d apprentissage SGD')
plt.xlabel('Nombre d etapes');
plt.ylabel('RMSE');
plt.legend()
plt.grid()
plt.show()

In [ ]:
"""
Le modèle semble fonctionner bien avec,relativement, une basse valeur de RMSE après convergence.
La performance du modèle peut dépendre des paramètres (gamma), (lambda) et k qu'on a varié à plusieurs reprises afin d'obtenir 
le meilleur RMSE.

Après cette étape, on peut comparer le rating réel avec le rating estimé; Pour ce faire, on utilise la matrice User-item qu'on a 
déjà calculée et utilisé la fonction prediction(P,Q) implémentée précédemment. 

"""

### 2.2.2 La généralisation de ce modèle:

In [ ]:
import numpy as np
import math
import random

random.seed(123)
np.random.seed(123)

P = 3 * np.random.rand(k, m)
Q = 3 * np.random.rand(k, n)

train_errors = []
test_errors = []

for step in range(steps):
    for u, i in zip(users, items):

        Pu = P[:, u].copy()
        Qi = Q[:, i].copy()

        e = data_matrix[u, i] - prediction(Pu, Qi)

        P[:, u] += gamma * (e * Qi - lmbda * Pu)
        Q[:, i] += gamma * (e * Pu - lmbda * Qi)

In [ ]:
R_pred=prediction(P,Q)
x = np.zeros((n_users, n_items))
for i in range(0,n_items):
    a=max(R_pred[:,i])
    b=min(R_pred[:,i])
    c=0
    d=5
    for j in range(0,n_users):
        x[j,i]=(R_pred[:,i][j]-(a-c))*d/(b-a+c)
        if math.isnan(x[j,i]): x[j,i]=0
            


In [ ]:
#Recommander des items pour un utilisateur donné:
import pandas as pd

x = pd.DataFrame(x)

def getrecom_modbased_sgd(iduser, n=10, ch="all"):
    user_real = R.loc[iduser-1]
    user_pred = x.loc[iduser-1]

    estim = user_pred[user_real == 0]   # items not rated
    donne = user_real[user_real > 0]    # already rated

    res = estim if ch == "discover" else pd.concat([estim, donne])

    return res.sort_values(ascending=False).head(n)

In [ ]:
#recommander l'utilisateur 18 en prenons en compte tous les items y compris ceux qu'il a déjà noté. Afficher les 10 premiers résultats
getrecom_modbased_sgd(18,10,"all")

In [ ]:
#recommander l'utilisateur 18 en prenons en compte que les items qu'il n'a pas noté. Afficher les 10 premiers résultats
getrecom_modbased_sgd(18,10,"discover")